# Exploration (lightweight)

> **SIMULATED DATA.** Quick EDA of the processed cohort. All real analysis lives in `src/` and is run by `make all`; this notebook only *reads* its outputs.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src import config
m = pd.read_parquet(config.PROCESSED_DIR / 'metrics_per_player.parquet')
print(f'{len(m):,} players in the analysis cohort')
m.head()

78,901 players in the analysis cohort


,player_id,variant,install_date,enrolment_week,platform,device_tier,country_tier,acquisition_channel,d1_retained,d7_retained,tutorial_complete,levels_by_d3,purchased_d7,revenue_d7,crashed_d7,avg_session_len,sessions_d7,day0_sessions,revenue_d7_d34
0,1002086,treatment,2026-09-01,1,iOS,high,T1,cross_promo,0,0,1,2,0,0.0,0,260.000000,2,1,0.0
1,1002087,treatment,2026-09-01,1,Android,low,T2,organic,0,0,0,0,0,0.0,0,127.500000,2,2,0.0
2,1002088,control,2026-09-01,1,Android,high,T2,cross_promo,0,0,0,6,0,0.0,0,265.000000,2,2,0.0
3,1002089,treatment,2026-09-01,1,Android,low,T2,organic,0,0,0,0,0,0.0,0,290.333333,3,1,0.0
4,1002090,control,2026-09-01,1,Android,low,T1,organic,0,0,1,4,0,0.0,0,93.000000,1,1,0.0


## Heavy tails: why revenue needs bootstrap / Mann-Whitney checks

In [2]:
payers = m[m.revenue_d7 > 0].revenue_d7.sort_values(ascending=False)
top1 = payers.head(max(1, len(payers)//100)).sum() / payers.sum()
print(f'payers: {len(payers):,} ({len(payers)/len(m):.2%}); top 1% of payers = {top1:.1%} of first-week revenue')
m[['revenue_d7', 'avg_session_len', 'levels_by_d3']].describe(percentiles=[.5, .9, .99]).round(2)

payers: 1,888 (2.39%); top 1% of payers = 11.8% of first-week revenue


,revenue_d7,avg_session_len,levels_by_d3
count,78901.00,76332.00,78901.0
mean,0.19,329.74,6.9
std,2.50,275.17,6.9
min,0.00,5.00,0.0
50%,0.00,282.00,5.0
90%,0.00,558.75,16.0
99%,4.99,1234.00,32.0
max,103.97,24450.00,74.0


## Day-0 sessions vs D7 retention (the CUPED covariate)

In [3]:
m.groupby(m.day0_sessions.clip(upper=6))['d7_retained'].agg(['mean', 'size']).rename(index={6: '6+'}).round(3)

,mean,size
day0_sessions,,
0,0.000,2569
1,0.103,22447
2,0.152,24320
3,0.221,15835
4,0.294,7897
5,0.405,3526
6+,0.552,2307


## Where the lift comes from: D7 retention by tutorial completion (descriptive, not causal)

In [4]:
m.pivot_table(index='tutorial_complete', columns='variant', values='d7_retained', aggfunc='mean').round(3)

variant,control,treatment
tutorial_complete,,
0,0.123,0.111
1,0.215,0.213
